# Intrusion Detection System using Machine Learning
## AIML Assignment 2

**Objective:** Build and compare multiple Machine Learning models for Network Intrusion Detection using the KDD Cup 1999 dataset.

**Dataset:** KDD Cup 1999 — a benchmark dataset for intrusion detection containing ~4.9 million network connection records with 41 features and a label indicating normal or attack type.

**Attack Categories:**
| Category | Description | Examples |
|----------|-------------|----------|
| DoS | Denial of Service | smurf, neptune, back, teardrop, pod, land |
| Probe | Surveillance/Probing | portsweep, ipsweep, nmap, satan |
| R2L | Remote to Local | warezclient, guess_passwd, warezmaster, ftp_write |
| U2R | User to Root | buffer_overflow, rootkit, loadmodule, perl |
| Normal | Legitimate traffic | normal |

**Models Implemented:**
1. Naive Bayes
2. Decision Tree
3. Random Forest
4. Support Vector Machine (SVM)
5. Logistic Regression
6. Gradient Boosting

---
## 1. Environment Setup & Data Loading

In [ ]:
# Install kagglehub if not already installed
!pip install -q kagglehub

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

import time

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("All libraries imported successfully.")

In [ ]:
import kagglehub

# Download latest version of KDD Cup 1999 dataset
path = kagglehub.dataset_download("kavl31/kdd-cup-1999-data")
print("Path to dataset files:", path)

# List files in the downloaded directory
for f in os.listdir(path):
    fpath = os.path.join(path, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f} — {size_mb:.1f} MB")

In [ ]:
# KDD Cup 1999 column names
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label'
]

# Find the main data file (try common KDD file names)
data_file = None
for candidate in ['kddcup.data_10_percent.gz', 'kddcup.data_10_percent', 
                   'kddcup.data.gz', 'kddcup.data',
                   'kddcup.data_10_percent_corrected', 'kddcup.data.corrected']:
    candidate_path = os.path.join(path, candidate)
    if os.path.exists(candidate_path):
        data_file = candidate_path
        print(f"Found data file: {candidate}")
        break

# If specific file not found, pick the largest CSV-like file
if data_file is None:
    all_files = [(os.path.join(path, f), os.path.getsize(os.path.join(path, f)))
                 for f in os.listdir(path) if not f.startswith('.')]
    all_files.sort(key=lambda x: x[1], reverse=True)
    data_file = all_files[0][0]
    print(f"Using largest file: {os.path.basename(data_file)}")

# Load the dataset
df = pd.read_csv(data_file, header=None, names=columns)
print(f"\nDataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical summary of numerical features
df.describe().T

In [ ]:
# Clean the label column (remove trailing '.' if present)
df['label'] = df['label'].str.strip().str.rstrip('.')

# Distribution of attack labels
print("Label distribution:")
label_counts = df['label'].value_counts()
print(label_counts)
print(f"\nTotal unique labels: {df['label'].nunique()}")

In [ ]:
# Map specific attack types to broader categories
attack_map = {
    'normal': 'Normal',
    # DoS attacks
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS', 'processtable': 'DoS',
    'udpstorm': 'DoS', 'apache2': 'DoS', 'worm': 'DoS',
    # Probe attacks
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    # R2L attacks
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'httptunnel': 'R2L',
    # U2R attacks
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
    'rootkit': 'U2R', 'xterm': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R',
}

df['attack_category'] = df['label'].map(attack_map)
# Any unmapped labels go to 'Unknown'
df['attack_category'] = df['attack_category'].fillna('Unknown')

print("Attack Category Distribution:")
cat_counts = df['attack_category'].value_counts()
print(cat_counts)
print(f"\nUnknown labels: {df[df['attack_category']=='Unknown']['label'].unique()}")

In [ ]:
# Visualize attack category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']
cat_counts.plot(kind='bar', ax=axes[0], color=colors[:len(cat_counts)], edgecolor='black')
axes[0].set_title('Distribution of Attack Categories', fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(cat_counts.values):
    axes[0].text(i, v + max(cat_counts)*0.01, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=colors[:len(cat_counts)], startangle=140, pctdistance=0.85)
axes[1].set_title('Proportion of Attack Categories', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of categorical features
cat_features = ['protocol_type', 'service', 'flag']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(cat_features):
    top_vals = df[feat].value_counts().head(10)
    top_vals.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'{feat} (top 10)', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## 3. Data Pre-processing

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_info = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Missing values per column:")
print(missing_info[missing_info['Missing Count'] > 0])
if missing.sum() == 0:
    print("No missing values found in the dataset.")

In [ ]:
# Check for duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count:,} ({dup_count/len(df)*100:.2f}%)")

# Remove duplicates
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

In [ ]:
# Create binary label: Normal vs Attack
df['binary_label'] = df['attack_category'].apply(lambda x: 0 if x == 'Normal' else 1)

print("Binary label distribution:")
print(df['binary_label'].value_counts().rename({0: 'Normal', 1: 'Attack'}))
print(f"\nAttack ratio: {df['binary_label'].mean():.2%}")

In [ ]:
# Encode categorical features using LabelEncoder
label_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
# Exclude label columns from encoding
categorical_cols = [c for c in categorical_cols if c not in ['label', 'attack_category']]

print("Encoding categorical features:")
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"  {col}: {len(le.classes_)} unique values")

print("\nAll categorical features encoded.")

In [ ]:
# Prepare feature matrix and target
# Drop label columns
feature_cols = [c for c in df.columns if c not in ['label', 'attack_category', 'binary_label']]
X = df[feature_cols].copy()
y = df['binary_label'].copy()  # Binary classification: Normal (0) vs Attack (1)

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts().rename({0: 'Normal', 1: 'Attack'})}")

In [ ]:
# Sampling for computational efficiency
# The full dataset can be very large; use stratified sampling if needed
SAMPLE_SIZE = 100000  # Adjust based on available resources

if len(X) > SAMPLE_SIZE:
    print(f"Dataset is large ({len(X):,} rows). Using stratified sample of {SAMPLE_SIZE:,} rows.")
    X_sampled, _, y_sampled, _ = train_test_split(
        X, y, train_size=SAMPLE_SIZE, stratify=y, random_state=42
    )
    X = X_sampled.reset_index(drop=True)
    y = y_sampled.reset_index(drop=True)
    print(f"Sampled shape: {X.shape}")
    print(f"Sampled target distribution:\n{y.value_counts().rename({0: 'Normal', 1: 'Attack'})}")
else:
    print(f"Dataset size ({len(X):,}) is manageable. Using full dataset.")

---
## 4. Data Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = X.corr()

plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, linewidths=0.5,
            fmt='.1f', square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Find highly correlated feature pairs (|r| > 0.9)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': corr_matrix.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False)
print(f"Highly correlated pairs (|r| > 0.9): {len(high_corr_df)}")
high_corr_df

In [ ]:
# Correlation of each feature with the target variable
target_corr = X.corrwith(y).abs().sort_values(ascending=False)

plt.figure(figsize=(14, 8))
target_corr.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Feature Correlation with Target (|r|)', fontsize=14, fontweight='bold')
plt.ylabel('Absolute Correlation')
plt.axhline(y=0.1, color='red', linestyle='--', label='Threshold (0.1)')
plt.legend()
plt.tight_layout()
plt.show()

print("\nTop 15 features by correlation with target:")
print(target_corr.head(15))

---
## 5. Feature Selection

In [ ]:
# Method 1: Remove highly correlated features (keep one from each correlated pair)
# Identify features to drop from highly correlated pairs
features_to_drop = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            # Drop the feature with lower correlation to target
            feat_i = corr_matrix.columns[i]
            feat_j = corr_matrix.columns[j]
            corr_i = abs(X[feat_i].corr(y))
            corr_j = abs(X[feat_j].corr(y))
            if corr_i < corr_j:
                features_to_drop.add(feat_i)
            else:
                features_to_drop.add(feat_j)

print(f"Features to drop (corr > 0.95): {len(features_to_drop)}")
print(sorted(features_to_drop))

In [ ]:
# Method 2: SelectKBest using mutual information
selector = SelectKBest(score_func=mutual_info_classif, k='all')
selector.fit(X, y)

mi_scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(14, 8))
mi_scores.plot(kind='bar', color='darkorange', edgecolor='black')
plt.title('Mutual Information Scores (Feature Importance)', fontsize=14, fontweight='bold')
plt.ylabel('MI Score')
plt.tight_layout()
plt.show()

print("\nTop 20 features by Mutual Information:")
print(mi_scores.head(20))

In [ ]:
# Select top K features based on mutual information
K = 25  # Number of features to select
selected_features = mi_scores.head(K).index.tolist()

# Also remove highly correlated features from the selected set
selected_features = [f for f in selected_features if f not in features_to_drop]

print(f"Final selected features ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f} (MI: {mi_scores[f]:.4f})")

X_selected = X[selected_features].copy()

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_selected),
    columns=X_selected.columns
)

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"\nScaled features summary:")
X_scaled.describe().loc[['mean', 'std', 'min', 'max']]

In [ ]:
# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set:     {X_test.shape[0]:,} samples")
print(f"\nTraining target distribution:\n{y_train.value_counts().rename({0: 'Normal', 1: 'Attack'})}")
print(f"\nTest target distribution:\n{y_test.value_counts().rename({0: 'Normal', 1: 'Attack'})}")

---
## 6. Model Building & Training

We train six models as required:
1. **Gaussian Naive Bayes** — probabilistic classifier assuming feature independence
2. **Decision Tree** — rule-based splitting classifier
3. **Random Forest** — ensemble of decision trees with bagging
4. **Support Vector Machine (SVM)** — maximum-margin classifier
5. **Logistic Regression** — linear model for binary classification
6. **Gradient Boosting** — sequential ensemble that corrects errors iteratively

In [ ]:
# Define all models
models = {
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=15, min_samples_split=10, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=5,
        n_jobs=-1, random_state=42
    ),
    'SVM': SVC(
        kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000, solver='lbfgs', random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42
    )
}

print(f"Models to train: {len(models)}")
for name in models:
    print(f"  - {name}")

In [ ]:
# Train all models and collect metrics
results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    
    # Train
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predict
    start_time = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start_time
    
    # Probabilities for ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.decision_function(X_test)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'train_time': train_time,
        'predict_time': predict_time
    }
    
    print(f"  Accuracy:     {acc:.4f}")
    print(f"  Precision:    {prec:.4f}")
    print(f"  Recall:       {rec:.4f}")
    print(f"  F1-Score:     {f1:.4f}")
    print(f"  ROC-AUC:      {roc_auc:.4f}")
    print(f"  Train time:   {train_time:.2f}s")
    print(f"  Predict time: {predict_time:.4f}s")

print(f"\n{'='*60}")
print("All models trained successfully!")
print(f"{'='*60}")

---
## 7. Validation & Comparison

In [ ]:
# Summary comparison table
comparison_data = []
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1-Score': res['f1_score'],
        'ROC-AUC': res['roc_auc'],
        'Train Time (s)': res['train_time'],
        'Predict Time (s)': res['predict_time']
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Model')
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
comparison_df.style.format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'ROC-AUC': '{:.4f}',
    'Train Time (s)': '{:.2f}', 'Predict Time (s)': '{:.4f}'
})

In [ ]:
# Visual comparison of metrics
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
plot_df = comparison_df[metrics_to_plot]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(plot_df.index))
width = 0.15
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i * width, plot_df[metric], width,
                  label=metric, color=colors[i], edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(plot_df.index, rotation=30, ha='right')
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
    axes[idx].set_title(f'{name}\n(Acc: {res["accuracy"]:.4f})', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves for all models
plt.figure(figsize=(10, 8))
colors_roc = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=colors_roc[idx], lw=2,
             label=f"{name} (AUC = {res['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification reports
for name, res in results.items():
    print(f"\n{'='*60}")
    print(f"Classification Report: {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, res['y_pred'],
                                target_names=['Normal', 'Attack']))

In [ ]:
# Cross-Validation (5-fold) for robust evaluation
print("5-Fold Stratified Cross-Validation Results:")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='f1_weighted', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:25s} — Mean F1: {scores.mean():.4f} ± {scores.std():.4f}  "
          f"[{', '.join([f'{s:.4f}' for s in scores])}]")

# Box plot of CV results
plt.figure(figsize=(12, 6))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(column=list(cv_results.keys()), vert=True, patch_artist=True,
              boxprops=dict(facecolor='lightblue', color='navy'),
              medianprops=dict(color='red', linewidth=2))
plt.title('5-Fold Cross-Validation F1-Scores', fontsize=14, fontweight='bold')
plt.ylabel('F1-Score (Weighted)')
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Training time comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(results.keys())
train_times = [results[m]['train_time'] for m in model_names]
predict_times = [results[m]['predict_time'] for m in model_names]

axes[0].barh(model_names, train_times, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_title('Training Time', fontweight='bold')
for i, v in enumerate(train_times):
    axes[0].text(v + max(train_times)*0.02, i, f'{v:.2f}s', va='center')

axes[1].barh(model_names, predict_times, color='coral', edgecolor='black')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Prediction Time', fontweight='bold')
for i, v in enumerate(predict_times):
    axes[1].text(v + max(predict_times)*0.02, i, f'{v:.4f}s', va='center')

plt.suptitle('Computational Cost Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Feature Importance Analysis

In [ ]:
# Feature importance from tree-based models
tree_models = ['Decision Tree', 'Random Forest', 'Gradient Boosting']

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for idx, name in enumerate(tree_models):
    model = results[name]['model']
    importances = pd.Series(
        model.feature_importances_, index=selected_features
    ).sort_values(ascending=True)
    
    importances.tail(15).plot(
        kind='barh', ax=axes[idx], color='teal', edgecolor='black'
    )
    axes[idx].set_title(f'{name}\nTop 15 Features', fontweight='bold')
    axes[idx].set_xlabel('Importance')

plt.suptitle('Feature Importance from Tree-Based Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Multi-class Classification (Attack Category)

Beyond binary classification, we also evaluate the best-performing model on the 5-class attack category problem.

In [ ]:
# Encode attack categories for multi-class classification
le_attack = LabelEncoder()
y_multi = le_attack.fit_transform(df['attack_category'])

# If we sampled earlier, apply same sampling
if len(X_selected) < len(df):
    y_multi_series = pd.Series(y_multi)
    # Re-sample consistently
    np.random.seed(42)
    sample_idx = X_selected.index
    y_multi = y_multi_series.iloc[sample_idx].values

# Use the same selected features, re-scale
X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_scaled, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

# Train Random Forest for multi-class
rf_multi = RandomForestClassifier(
    n_estimators=100, max_depth=20, n_jobs=-1, random_state=42
)
rf_multi.fit(X_multi_train, y_multi_train)
y_multi_pred = rf_multi.predict(X_multi_test)

print("Multi-class Classification Report (Random Forest):")
print("=" * 60)
print(classification_report(
    y_multi_test, y_multi_pred,
    target_names=le_attack.classes_
))

# Confusion matrix
plt.figure(figsize=(8, 6))
cm_multi = confusion_matrix(y_multi_test, y_multi_pred)
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=le_attack.classes_, yticklabels=le_attack.classes_)
plt.title('Multi-class Confusion Matrix (Random Forest)', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## 10. Conclusion

### Key Findings

**Data Pre-processing:**
- The KDD Cup 1999 dataset contains 41 features with 3 categorical columns (protocol_type, service, flag)
- Duplicate rows were identified and removed
- Label encoding was applied to categorical features; StandardScaler was used for numerical normalization

**Feature Selection:**
- Highly correlated feature pairs (|r| > 0.95) were identified and redundant features removed
- Mutual Information scoring was used to rank and select the most informative features
- The final feature set balances informativeness with minimal redundancy

**Model Comparison:**

| Aspect | Observation |
|--------|-------------|
| Best Accuracy | Tree-based ensemble models (Random Forest, Gradient Boosting) typically achieve the highest accuracy |
| Best Interpretability | Decision Tree provides transparent, rule-based decisions |
| Fastest Training | Naive Bayes and Logistic Regression are the fastest to train |
| Best Overall | Random Forest / Gradient Boosting offer the best trade-off between accuracy and robustness |
| SVM | Performs well but is computationally expensive on large datasets |
| Naive Bayes | Fastest but may underperform due to the independence assumption |

**Recommendations:**
- For production IDS: **Random Forest** or **Gradient Boosting** — highest detection rates with manageable false positive rates
- For real-time detection with resource constraints: **Decision Tree** — fast inference with good accuracy
- For explainability requirements: **Logistic Regression** or **Decision Tree** — easy to interpret and audit

**Innovation:**
- Multi-class attack category prediction (5-class) in addition to binary classification
- Comprehensive evaluation using multiple metrics (Accuracy, Precision, Recall, F1, ROC-AUC)
- 5-fold stratified cross-validation for robust performance estimation
- Feature importance analysis across tree-based models for interpretability